In [ ]:

# ============================================================
# E-Commerce Dataset | Data Cleaning & Preprocessing Pipeline
# Google Colab | Customers, Orders, Order_items, Products
# ============================================================

import pandas as pd
import numpy as np
import re
from google.colab import files

# ------------------------------------------------------------
# 1. Upload Excel file
# ------------------------------------------------------------
uploaded = files.upload()
input_file = next(iter(uploaded.keys()))

# ------------------------------------------------------------
# 2. Configuration
# ------------------------------------------------------------
EXPECTED_SHEETS = [
    "Customers",
    "Order_items",
    "Orders",
    "Products"
]

NUMERIC_COLS = {
    "Customers": [
        "customer_zip_code_prefix",
        "geolocation_zip_code_prefix"
    ],
    "Order_items": [
        "quantity",
        "unit_price",
        "unit_freight_value"
    ],
    "Orders": [
        "payment_installments",
        "payment_value"
    ],
    "Products": [
        "product_name_lenght",
        "product_description_lenght",
        "product_photos_qty",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    ]
}

DATE_COLS = {
    "Order_items": ["shipping_limit_date"],
    "Orders": [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
}

# Columns expected to uniquely identify records.
PRIMARY_KEYS = {
    "Customers": ["customer_id"],
    "Order_items": [],
    "Orders": ["order_id"],
    "Products": ["product_id"]
}

# ------------------------------------------------------------
# 3. Read Excel safely
# Read as strings first to protect IDs and postal codes.
# ------------------------------------------------------------
excel = pd.ExcelFile(input_file)

missing_sheets = set(EXPECTED_SHEETS) - set(excel.sheet_names)
if missing_sheets:
    raise ValueError(
        f"Missing expected sheets: {sorted(missing_sheets)}\n"
        f"Available sheets: {excel.sheet_names}"
    )

raw_data = {}
cleaned_data = {}
quality_summary = []
column_reports = []
duplicate_key_reports = []
issue_reports = []

for sheet in EXPECTED_SHEETS:
    df = pd.read_excel(
        input_file,
        sheet_name=sheet,
        dtype=str
    )

    # Preserve a copy for before/after comparisons.
    original = df.copy()
    original_rows = len(df)

    # Remove completely empty rows and columns.
    df = df.dropna(how="all")
    df = df.dropna(axis=1, how="all")

    # Standardize column names.
    df.columns = [
        re.sub(r"_+", "_",
               re.sub(r"[^a-zA-Z0-9]+", "_", str(c).strip().lower())
        ).strip("_")
        for c in df.columns
    ]

    # Standardize missing-value representations.
    missing_tokens = {
        "", "na", "n/a", "null", "none", "nan", "nat"
    }

    for col in df.select_dtypes(include="object").columns:
        s = df[col].astype("string").str.strip()
        df[col] = s.mask(
            s.str.lower().isin(missing_tokens),
            pd.NA
        )

    # Normalize text formatting.
    # IDs are NOT lowercased or otherwise transformed.
    id_cols = {
        "customer_id", "customer_unique_id",
        "order_id", "product_id"
    }

    text_cols = [
        c for c in df.select_dtypes(include="object").columns
        if c not in id_cols
        and c not in NUMERIC_COLS.get(sheet, [])
        and c not in DATE_COLS.get(sheet, [])
    ]

    for col in text_cols:
        df[col] = (
            df[col].astype("string")
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
            .str.lower()
        )

    # Convert numeric columns.
    for col in NUMERIC_COLS.get(sheet, []):
        if col in df.columns:
            # Handle decimal commas where present.
            s = df[col].astype("string").str.strip()
            s = s.str.replace(",", ".", regex=False)
            converted = pd.to_numeric(s, errors="coerce")

            # Record non-empty values that failed conversion.
            bad = s.notna() & converted.isna()
            if bad.any():
                issue_reports.append(pd.DataFrame({
                    "sheet": sheet,
                    "issue": "invalid_numeric_value",
                    "column": col,
                    "excel_row": df.index[bad] + 2,
                    "value": s[bad].astype(str).values
                }))

            df[col] = converted

    # Convert date/time columns.
    for col in DATE_COLS.get(sheet, []):
        if col in df.columns:
            s = df[col].astype("string").str.strip()
            converted = pd.to_datetime(s, errors="coerce")

            bad = s.notna() & converted.isna()
            if bad.any():
                issue_reports.append(pd.DataFrame({
                    "sheet": sheet,
                    "issue": "invalid_datetime",
                    "column": col,
                    "excel_row": df.index[bad] + 2,
                    "value": s[bad].astype(str).values
                }))

            df[col] = converted

    # Remove exact duplicate rows only.
    # Do not deduplicate based on a non-unique business key.
    duplicate_count = int(df.duplicated().sum())
    if duplicate_count:
        df = df.drop_duplicates().copy()

    # --------------------------------------------------------
    # 4. Data quality checks
    # --------------------------------------------------------
    for col in df.columns:
        column_reports.append({
            "sheet": sheet,
            "column": col,
            "dtype": str(df[col].dtype),
            "rows": len(df),
            "missing_count": int(df[col].isna().sum()),
            "missing_percent": round(
                df[col].isna().mean() * 100, 2
            ),
            "unique_values": int(df[col].nunique(dropna=True)
            ),
            "duplicate_values": int(
                df[col].notna().sum() -
                df[col].nunique(dropna=True)
            )
        })

    # Check primary-key duplicates and missing keys.
    for key in PRIMARY_KEYS.get(sheet, []):
        if key in df.columns:
            duplicate_mask = (
                df[key].notna() & df[key].duplicated(keep=False)
            )

            duplicate_key_reports.append({
                "sheet": sheet,
                "key_column": key,
                "missing_key_count": int(df[key].isna().sum()),
                "duplicate_key_rows": int(duplicate_mask.sum()),
                "unique_key_values": int(df[key].nunique())
            })

            if duplicate_mask.any():
                issue_reports.append(pd.DataFrame({
                    "sheet": sheet,
                    "issue": "duplicate_primary_key",
                    "column": key,
                    "excel_row": df.index[duplicate_mask] + 2,
                    "value": df.loc[duplicate_mask, key].astype(str).values
                }))

    quality_summary.append({
        "sheet": sheet,
        "original_rows": original_rows,
        "cleaned_rows": len(df),
        "removed_exact_duplicates": duplicate_count,
        "columns_after_cleaning": len(df.columns),
        "missing_cells_after_cleaning": int(df.isna().sum().sum())
    })

    raw_data[sheet] = original
    cleaned_data[sheet] = df.reset_index(drop=True)

# ------------------------------------------------------------
# 5. Domain-specific validation
# Flag suspicious values; do not silently delete records.
# ------------------------------------------------------------
def flag_invalid(df, sheet, col, condition, description):
    if col not in df.columns:
        return

    mask = df[col].notna() & condition(df[col])

    if mask.any():
        issue_reports.append(pd.DataFrame({
            "sheet": sheet,
            "issue": description,
            "column": col,
            "excel_row": df.index[mask] + 2,
            "value": df.loc[mask, col].astype(str).values
        }))

# Prices, quantities, freight, and payment values should
# normally be non-negative. Zero is flagged for review too
# only where a strictly positive value is expected.
for sheet, col in [
    ("Order_items", "quantity"),
    ("Order_items", "unit_price"),
    ("Order_items", "unit_freight_value"),
    ("Orders", "payment_value"),
    ("Products", "product_weight_g"),
    ("Products", "product_length_cm"),
    ("Products", "product_height_cm"),
    ("Products", "product_width_cm"),
    ("Products", "product_photos_qty")
]:
    df = cleaned_data[sheet]
    flag_invalid(
        df, sheet, col,
        lambda s: s < 0,
        "negative_value_review"
    )

# Quantity and photo count are expected to be whole numbers.
for sheet, col in [
    ("Order_items", "quantity"),
    ("Products", "product_photos_qty"),
    ("Products", "product_name_lenght"),
    ("Products", "product_description_lenght")
]:
    df = cleaned_data[sheet]
    flag_invalid(
        df, sheet, col,
        lambda s: (s % 1) != 0,
        "non_integer_count_review"
    )

# Installments are expected to be positive whole numbers.
# Keep original values and flag anomalies for investigation.
df = cleaned_data["Orders"]
flag_invalid(
    df, "Orders", "payment_installments",
    lambda s: (s <= 0) | ((s % 1) != 0),
    "invalid_installment_count_review"
)

# Check date logic without automatically changing dates.
df = cleaned_data["Orders"]

date_pairs = [
    ("order_approved_at", "order_purchase_timestamp"),
    ("order_delivered_carrier_date", "order_approved_at"),
    ("order_delivered_customer_date", "order_delivered_carrier_date")
]

for later, earlier in date_pairs:
    if later in df.columns and earlier in df.columns:
        mask = (
            df[later].notna()
            & df[earlier].notna()
            & (df[later] < df[earlier])
        )

        if mask.any():
            issue_reports.append(pd.DataFrame({
                "sheet": "Orders",
                "issue": "chronological_date_inconsistency",
                "column": later,
                "excel_row": df.index[mask] + 2,
                "value": (
                    df.loc[mask, earlier].astype(str)
                    + " -> "
                    + df.loc[mask, later].astype(str)
                ).values
            }))

# ------------------------------------------------------------
# 6. Referential integrity checks
# ------------------------------------------------------------
def check_relationship(child_sheet, child_col,
                       parent_sheet, parent_col):
    child = cleaned_data[child_sheet]
    parent = cleaned_data[parent_sheet]

    if child_col not in child.columns or parent_col not in parent.columns:
        return

    parent_keys = set(parent[parent_col].dropna().astype(str))
    mask = (
        child[child_col].notna()
        & ~child[child_col].astype(str).isin(parent_keys)
    )

    if mask.any():
        issue_reports.append(pd.DataFrame({
            "sheet": child_sheet,
            "issue": f"unmatched_reference_to_{parent_sheet}",
            "column": child_col,
            "excel_row": child.index[mask] + 2,
            "value": child.loc[mask, child_col].astype(str).values
        }))

check_relationship(
    "Orders", "customer_id", "Customers", "customer_id"
)
check_relationship(
    "Order_items", "order_id", "Orders", "order_id"
)
check_relationship(
    "Order_items", "product_id", "Products", "product_id"
)

# ------------------------------------------------------------
# 7. Build reports
# ------------------------------------------------------------
quality_df = pd.DataFrame(quality_summary)
columns_df = pd.DataFrame(column_reports)
keys_df = pd.DataFrame(duplicate_key_reports)

if issue_reports:
    issues_df = pd.concat(issue_reports, ignore_index=True)
else:
    issues_df = pd.DataFrame(columns=[
        "sheet", "issue", "column", "excel_row", "value"
    ])

# ------------------------------------------------------------
# 8. Export cleaned workbook and reports
# ------------------------------------------------------------
cleaned_file = "ecommerce_cleaned.xlsx"
report_file = "ecommerce_quality_report.xlsx"

with pd.ExcelWriter(cleaned_file, engine="openpyxl") as writer:
    for sheet, df in cleaned_data.items():
        df.to_excel(writer, sheet_name=sheet, index=False)

with pd.ExcelWriter(report_file, engine="openpyxl") as writer:
    quality_df.to_excel(writer, sheet_name="Sheet_Summary", index=False)
    columns_df.to_excel(writer, sheet_name="Column_Quality", index=False)
    keys_df.to_excel(writer, sheet_name="Primary_Key_Checks", index=False)
    issues_df.to_excel(writer, sheet_name="Issues_To_Review", index=False)

# ------------------------------------------------------------
# 9. Display summary and download files
# ------------------------------------------------------------
print("=" * 65)
print("DATA CLEANING COMPLETED")
print("=" * 65)

print("\nSHEET SUMMARY:")
display(quality_df)

print("\nPRIMARY KEY CHECKS:")
display(keys_df)

print("\nISSUES FOUND:")
print(f"Total issue rows: {len(issues_df):,}")
display(issues_df.head(20))

print("\nMissing values by column:")
display(
    columns_df.sort_values(
        ["sheet", "missing_percent"],
        ascending=[True, False]
    )
)

files.download(cleaned_file)
files.download(report_file)


Saving New Microsoft Excel Worksheet.xlsx to New Microsoft Excel Worksheet.xlsx
DATA CLEANING COMPLETED

SHEET SUMMARY:


,sheet,original_rows,cleaned_rows,removed_exact_duplicates,columns_after_cleaning,missing_cells_after_cleaning
0,Customers,99441,99441,0,8,834
1,Order_items,102425,102425,0,6,0
2,Orders,99441,99441,0,11,4911
3,Products,32951,32951,0,10,3071



PRIMARY KEY CHECKS:


,sheet,key_column,missing_key_count,duplicate_key_rows,unique_key_values
0,Customers,customer_id,0,0,99441
1,Orders,order_id,0,0,99441
2,Products,product_id,0,0,32951



ISSUES FOUND:
Total issue rows: 76,316


...

,sheet,column,dtype,rows,missing_count,missing_percent,unique_values,duplicate_values
5,Customers,geolocation_zip_code_prefix,Int64,99441,278,0.28,14837,84326
6,Customers,geolocation_city,string,99441,278,0.28,4159,95004
7,Customers,geolocation_state,string,99441,278,0.28,27,99136
0,Customers,customer_id,string,99441,0,0.00,99441,0
1,Customers,customer_unique_id,string,99441,0,0.00,96096,3345
2,Customers,customer_zip_code_prefix,Int64,99441,0,0.00,14994,84447
3,Customers,customer_city,string,99441,0,0.00,4119,95322
4,Customers,customer_state,string,99441,0,0.00,27,99414
8,Order_items,order_id,string,102425,0,0.00,98666,3759
9,Order_items,product_id,string,102425,0,0.00,32951,69474


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>